# Análise Espacial Utilizando o Python

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium
import mapclassify
import branca.colormap as cm
import os
import numpy as np

In [ ]:
# Leitura dos dados

os.chdir(r"....") # Local onde estão os arquivos espaciais

localidade = gpd.read_file("distritos_sp.gpkg")
enderecos = gpd.read_file("pontos_sp.shp")

print(localidade.crs)
print(enderecos.crs)

# Visualização dos mapas

In [ ]:
# Visualizações iniciais

localidade.plot(edgecolor="black", figsize=(8,8)) # Distritos de São Paulo
plt.show()

enderecos.plot(markersize=1, figsize=(8,8)) # Pontos localizados em São Paulo - residências e pontos comerciais
plt.show()


In [ ]:
# Filtrar domicílios (apenas pontos que são residências

domicilios = enderecos[enderecos["COD_ESP"] == 1].copy()

print(domicilios.crs)

domicilios.plot(color="red", markersize=2)
plt.show()

In [ ]:
# Sobreposição - Pontos e Polígonos

ax = localidade.plot(facecolor="white",
                     edgecolor="black",
                     figsize=(8,8))

domicilios.plot(ax=ax,
                color="red",
                markersize=2)

plt.show()

In [ ]:
# Mudança de projeção - O resultados acima não está adequado

localidade3857 = localidade.to_crs(3857)
domicilios3857 = domicilios.to_crs(3857)

In [ ]:
# Sobreposição - Pontos e Polígonos

ax = localidade3857.plot(facecolor="white",
                     edgecolor="black",
                     figsize=(8,8))

domicilios3857.plot(ax=ax,
                color="red",
                markersize=2)

plt.show()

In [ ]:
# Contagem de domicílios por distrito

localidade3857 = localidade.to_crs(3857)
domicilios3857 = domicilios.to_crs(3857)

join = gpd.sjoin(
    domicilios3857,
    localidade3857,
    predicate="within",
    how="left")

contagem = join.groupby(join.index_right).size()

localidade3857["qtd_domicilios"] = (
    contagem.reindex(localidade3857.index)
    .fillna(0)
    .astype(int))

print(localidade3857["qtd_domicilios"].sum())
print(len(domicilios3857))

In [ ]:
# Domicílios fora dos distritos

fora = join[join["index_right"].isna()].copy()

print(len(fora))

In [ ]:
# Mapa Folium - Domicílios fora dos distritos

localidade4326 = localidade3857.to_crs(4326)
fora4326 = fora.to_crs(4326)

m = folium.Map(location=[-23.55,-46.63],
               zoom_start=10)

folium.GeoJson(localidade4326).add_to(m)

for _, row in fora4326.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y,row.geometry.x],
        radius=4,
        color="red",
        fill=True
    ).add_to(m)

m

In [ ]:
# Estatísticas - Quantidade de domicílios por distrito

print(localidade3857["qtd_domicilios"].describe())

In [ ]:
# Mapa contínuo - Mapa Coroplético

fig, ax = plt.subplots(figsize=(10,10))

localidade3857.plot(
    column="qtd_domicilios",
    cmap="RdYlBu_r",
    legend=True,
    ax=ax)

ax.set_title("Quantidade de domicílios por distrito")
plt.show()

In [ ]:
# Classificação Jenks

jenks = mapclassify.FisherJenks(
    localidade3857["qtd_domicilios"],
    k=7)

localidade3857["classe"] = jenks.yb

fig, ax = plt.subplots(figsize=(10,10))

localidade3857.plot(
    column="classe",
    cmap="RdYlBu_r",
    legend=True,
    categorical=True,
    ax=ax)

ax.set_title("Quantidade de domicílios por distrito (Jenks)")
plt.show()

In [ ]:
# Quantile

fig, ax = plt.subplots(figsize=(10,10))

localidade3857.plot(
    column="qtd_domicilios",
    scheme="Quantiles",
    k=5,
    cmap="viridis",
    legend=True,
    ax=ax)

plt.show()

In [ ]:
# Equal Interval

fig, ax = plt.subplots(figsize=(10,10))

localidade3857.plot(
    column="qtd_domicilios",
    scheme="EqualInterval",
    k=5,
    cmap="viridis",
    legend=True,
    ax=ax)

plt.show()

In [ ]:
# Leaflet com todos os domicílios

domicilios4326 = domicilios3857.to_crs(4326)

m = folium.Map(location=[-23.55,-46.63],
               zoom_start=10)

folium.GeoJson(localidade4326).add_to(m)

for _, row in domicilios4326.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y,row.geometry.x],
        radius=2,
        color="red",
        fill=True,
        popup=str(row["COD_ESP"])
    ).add_to(m)

m

In [ ]:
# Filtrar um distrido - Perdizes

perdizes = localidade4326[
    localidade4326["NM_DIST"]=="Perdizes"
]

m = folium.Map(location=[-23.54,-46.68],
               zoom_start=13)

folium.GeoJson(
    perdizes,
    tooltip="Perdizes"
).add_to(m)

m

In [ ]:
# Filtrar um distrido - Morumbi

morumbi = localidade4326[
    localidade4326["NM_DIST"]=="Morumbi"
]

m = folium.Map(location=[-23.60,-46.72],
               zoom_start=13)

folium.GeoJson(
    morumbi,
    tooltip="Morumbi"
).add_to(m)

m

In [ ]:
# Mapa Coroplético

cmap = cm.linear.viridis.scale(
    localidade4326["qtd_domicilios"].min(),
    localidade4326["qtd_domicilios"].max())

m = folium.Map(location=[-23.55,-46.63],
               zoom_start=10)

folium.GeoJson(
    localidade4326,
    style_function=lambda feat: {
        "fillColor": cmap(
            feat["properties"]["qtd_domicilios"]
        ),
        "color":"white",
        "weight":1,
        "fillOpacity":0.7
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NM_DIST","qtd_domicilios"]
    )
).add_to(m)

cmap.caption = "Quantidade de domicílios"
cmap.add_to(m)

m

# Manipulação de Dados Espaciais

In [ ]:
# Leitura dos dados

os.chdir(r"...") # Local onde estão os arquivos espaciais

localidade = gpd.read_file("distritos_sp.gpkg")
enderecos = gpd.read_file("pontos_sp.shp")

print(localidade.crs)
print(enderecos.crs)

# Visualização

localidade.plot(edgecolor="black", figsize=(8,8))
plt.show()

enderecos.plot(markersize=2)
plt.show()

In [ ]:
# Filtragem

domicilios = enderecos[enderecos["COD_ESP"] == 1].copy() # Apenas residências
estab_ensino = enderecos[enderecos["COD_ESP"] == 4].copy() # Apenas estabelecimentos de ensino
estab_saude = enderecos[enderecos["COD_ESP"] == 5].copy() # Apenas estabelecimentos de saúde

localidade3857 = localidade.to_crs(3857)
domicilios3857 = domicilios.to_crs(3857)

ax = localidade3857.plot(
    facecolor="white",
    edgecolor="black",
    figsize=(8,8))

domicilios3857.plot(
    ax=ax,
    color="blue",
    markersize=2)

plt.show()

In [ ]:
# Predicados espaciais

localidade31983 = localidade.to_crs(31983)
estab_saude31983 = estab_saude.to_crs(31983)
estab_ensino31983 = estab_ensino.to_crs(31983)

# Within

within = gpd.sjoin(
    estab_saude31983,
    localidade31983,
    predicate="within",
    how="left")

print(within.head())

In [ ]:
# Intersects

intersects = gpd.sjoin(
    estab_saude31983,
    localidade31983,
    predicate="intersects")

print(intersects.head())

In [ ]:
# Contains

contains = gpd.sjoin(
    localidade31983,
    estab_ensino31983,
    predicate="contains")

print(contains.head())

In [ ]:
# Overlay

overlay = gpd.overlay(
    estab_saude31983,
    localidade31983,
    how="intersection")

ax = overlay.plot(
    facecolor="white",
    edgecolor="black",
    figsize=(8,8))

overlay.plot(
    ax=ax,
    color="blue",
    markersize=2)

In [ ]:
# Difference

diferenca = gpd.overlay(
    estab_saude31983,
    localidade31983,
    how="difference")

ax = diferenca.plot(
    facecolor="white",
    edgecolor="black",
    figsize=(8,8))

diferenca.plot(
    ax=ax,
    color="blue",
    markersize=2)

In [ ]:
# União dos polígonos gerando um outro mapa

sp = localidade.dissolve()

ax = sp.plot(
    facecolor="white",
    edgecolor="black",
    figsize=(8,8))

sp.plot(
    ax=ax,
    color="blue",
    markersize=2)

In [ ]:
# Área

localidade3857["area_km2_1"] = (
    localidade3857.area / 1_000_000
)

print(localidade3857.head())

In [ ]:
# Área

localidade31983["area_km2_2"] = (
    localidade31983.area / 1_000_000
)

print(localidade31983.head()) # Projeções diferentes geram resultados distintos. Compare com os valores acima.

In [ ]:
# Centróides

centroides = (
    localidade31983[["NM_DIST","geometry"]]
    .copy())

centroides.geometry = centroides.centroid

print(centroides.head())

In [ ]:
# Buffer de 1000 metros

estab_ensino31983 = estab_ensino.to_crs(31983)

area_influencia = estab_ensino31983.copy()

area_influencia.geometry = (
    area_influencia.buffer(1000))

print(area_influencia.head())

In [ ]:
# Domicílios dentro do Buffer

inter_domicilios = gpd.overlay(
    domicilios.to_crs(31983),
    area_influencia,
    how="intersection"
)

duplicados = (
    inter_domicilios
    .groupby("ID_1")
    .size()
    .sort_values(ascending=False)
)

print(duplicados.head())

In [ ]:
# Buffer x Distritos

inter_buffer = gpd.overlay(
    area_influencia.to_crs(4326),
    localidade.to_crs(4326),
    how="intersection")

inter_buffer = (
    inter_buffer[
        ["ID","NM_DIST","geometry"]
    ])

print(inter_buffer.head())

In [ ]:
# Distâncias

ensino31983 = estab_ensino.to_crs(31983)
saude31983 = estab_saude.to_crs(31983)

distancias = ensino31983.geometry.apply(
    lambda g: saude31983.distance(g))

print(distancias)

In [ ]:
# Matriz de Distâncias

import numpy as np

matriz = np.zeros(
    (len(ensino31983), len(saude31983)))

for i, g1 in enumerate(ensino31983.geometry):

    matriz[i] = saude31983.distance(g1)

dist_pp = pd.DataFrame(
    matriz,
    index=ensino31983["ID"],
    columns=saude31983["ID"])

print(dist_pp.head())

# Análises Espaciais Avançadas

In [ ]:
# Bibliotecas adicionais

import numpy as np
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import gaussian_kde
from scipy.interpolate import griddata
from shapely.geometry import Polygon
from scipy.spatial import Voronoi
from scipy.spatial import cKDTree

In [ ]:
# Leitura dos dados

localidade = gpd.read_file("distritos_sp.gpkg").to_crs(31983)
domicilios = gpd.read_file("pontos_sp.shp")
domicilios = domicilios[domicilios["COD_ESP"]==1].to_crs(31983)
pontos_venda = gpd.read_file("pdv_sp.gpkg").to_crs(31983)

In [ ]:
# Mapa de densidade de Kernel - Mapa de calor

x=domicilios.geometry.x
y=domicilios.geometry.y
xy=np.vstack([x,y])
kde=gaussian_kde(xy)

xmin,ymin,xmax,ymax=domicilios.total_bounds
xx,yy=np.mgrid[xmin:xmax:200j,ymin:ymax:200j]
z=kde(np.vstack([xx.ravel(),yy.ravel()])).reshape(xx.shape)

plt.figure(figsize=(8,8))
plt.imshow(z.T,origin='lower',extent=[xmin,xmax,ymin,ymax])
localidade.boundary.plot(ax=plt.gca(),color='black')
plt.title("Kernel Density")
plt.colorbar()
plt.show()

In [ ]:
# Domicílios por km²

join=gpd.sjoin(domicilios,localidade,predicate='within')
cont=join.groupby('index_right').size()
localidade['qtd']=cont.reindex(localidade.index).fillna(0)
localidade['area_km2']=localidade.area/1e6
localidade['dens_km2']=localidade['qtd']/localidade['area_km2']
localidade.plot(column='dens_km2',legend=True,cmap='viridis')
plt.title("Domicílios por km²")
plt.show()

In [ ]:
# Índice de Moran

from libpysal.weights import Queen
from esda.moran import Moran

w=Queen.from_dataframe(localidade)
w.transform='r'
mi=Moran(localidade['dens_km2'],w)
print("Moran I:",mi.I)
print("p-valor:",mi.p_sim)

In [ ]:
# Índice de Moran Local (LISA)

from esda.moran import Moran_Local

lisa=Moran_Local(localidade['dens_km2'],w)
localidade['cluster']=lisa.q
localidade.plot(column='cluster',categorical=True,legend=True,cmap='Set1')
plt.title("Clusters LISA")
plt.show()

In [ ]:
# Polígonos de Voronoi

pts=np.array([(g.x,g.y) for g in pontos_venda.geometry])
vor=Voronoi(pts)

polys=[]
for region in vor.regions:
    if not region or -1 in region:
        continue
    polys.append(Polygon(vor.vertices[region]))

vor_gdf=gpd.GeoDataFrame(geometry=polys,crs=domicilios.crs)
vor_gdf = gpd.overlay(vor_gdf, localidade, how="intersection")
ax=localidade.boundary.plot(figsize=(8,8),color='black')
vor_gdf.plot(ax=ax,facecolor='none',edgecolor='blue',linewidth=0.3)
plt.title("Voronoi")
plt.show()

In [ ]:
# Mapa de interpolação IDW (Inverse Distance Weighting)

x = pontos_venda.geometry.x.values
y = pontos_venda.geometry.y.values
z = pontos_venda["FATURAM"].values

xmin, ymin, xmax, ymax = localidade.total_bounds

res = 200

grid_x, grid_y = np.meshgrid(
    np.linspace(xmin, xmax, res),
    np.linspace(ymin, ymax, res))

grid_points = np.column_stack((grid_x.ravel(), grid_y.ravel()))

# Potência do IDW

power = 2

# Número máximo de vizinhos utilizados

k = 12

tree = cKDTree(np.column_stack((x, y)))

dist, idx = tree.query(grid_points, k=k)

dist[dist == 0] = 1e-10
weights = 1 / (dist ** power)

idw = np.sum(weights * z[idx], axis=1) / np.sum(weights, axis=1)

idw = idw.reshape(grid_x.shape)

fig, ax = plt.subplots(figsize=(10,10))

im = ax.imshow(
    idw,
    origin="lower",
    extent=[xmin, xmax, ymin, ymax],
    cmap="viridis"
)

localidade.boundary.plot(ax=ax, color="black", linewidth=1)

pontos_venda.plot(
    ax=ax,
    color="red",
    markersize=15,
    edgecolor="white"
)

plt.colorbar(im, ax=ax, label="FATURAM")

ax.set_title("Interpolação IDW - FATURAM")

plt.show()

In [ ]:
# Krigagem Ordinária (Ordinary Kriging)

import matplotlib.pyplot as plt

from pykrige.ok import OrdinaryKriging

x = pontos_venda.geometry.x.values
y = pontos_venda.geometry.y.values

z = pontos_venda["FATURAM"].values

xmin, ymin, xmax, ymax = localidade.total_bounds

gridx = np.linspace(xmin, xmax, 200)
gridy = np.linspace(ymin, ymax, 200)

OK = OrdinaryKriging(
    x,
    y,
    z,
    variogram_model="exponential",
    verbose=False,
    enable_plotting=False)

# Você também pode testar:
    # variogram_model="linear"
    # variogram_model="gaussian"
    # variogram_model="spherical"

z_krig, ss = OK.execute(
    "grid",
    gridx,
    gridy)

fig, ax = plt.subplots(figsize=(10,10))

im = ax.imshow(
    z_krig,
    extent=(xmin, xmax, ymin, ymax),
    origin="lower",
    cmap="viridis")

localidade.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1)

pontos_venda.plot(
    ax=ax,
    color="red",
    markersize=20,
    edgecolor="white")

plt.colorbar(im, ax=ax, label="FATURAM")

ax.set_title("Krigagem Ordinária - FATURAM")

plt.show()

# Questões

In [ ]:
### 1. Sistema de Referência
Por que é necessário transformar os dados para um sistema de coordenadas projetadas (por exemplo, EPSG:3857 ou UTM) antes de calcular distâncias, buffers ou realizar interpolações?


### 2. Junção Espacial
Qual é o objetivo da função abaixo?

```python
gpd.sjoin(domicilios, localidade, predicate="within")
```


### 3. Buffer
Ao criar um buffer de 500 metros ao redor dos pontos de venda, o que representa o polígono gerado?


### 4. Kernel Density
O mapa de densidade Kernel tem como principal finalidade:


### 5. Voronoi
Os polígonos de Voronoi representam:


### 6. Interpolação IDW
Na interpolação IDW, aumentar o parâmetro de potência (`power`) faz com que:


### 7. Krigagem
Uma vantagem da Krigagem em relação ao IDW é que ela:


### 8. Moran Global
Um Índice de Moran Global positivo e estatisticamente significativo indica que:


### 9. Moran Local (LISA)
Em um mapa LISA, uma região classificada como **High–High (HH)** representa:


### 10. Interpretação de Resultados
Durante a análise espacial, qual técnica é mais indicada para responder à pergunta: "Existe dependência espacial entre os valores observados?"